In [ ]:
!git clone https://github.com/andyzoujm/representation-engineering.git
!cd representation-engineering


In [ ]:
cd representation-engineering

In [ ]:
!pip install -e .
import matplotlib.pyplot as plt
import torch
from tqdm import tqdm
import numpy as np
from transformers import AutoTokenizer

# Import our custom Mamba implementation
from repe.rep_reading_pipeline import Mamba2ReadingPipeline

In [ ]:
cd examples/honesty/

In [ ]:
# Import utility functions
from utils import honesty_function_dataset, plot_lat_scans, plot_detection_results
# Load tokenizer only - we won't use the transformer model
model_name_or_path = "state-spaces/mamba-2.8b-hf"
tokenizer = AutoTokenizer.from_pretrained(model_name_or_path, use_fast=True, padding_side="left", legacy=False)
tokenizer.pad_token_id = 0
# Configure Mamba settings
rep_token = -1
hidden_dim = 768  # Customize this based on your needs
num_layers = 4    # Customize this based on your needs
hidden_layers = list(range(-1, -num_layers, -1))
n_difference = 1
direction_method = 'pca'

# Initialize the Mamba pipeline
mamba_pipeline = Mamba2ReadingPipeline(
    tokenizer=tokenizer,
    hidden_dim=hidden_dim,
    num_layers=num_layers,
    ssm_state_dim=16
)
user_tag = "USER:"
assistant_tag = "ASSISTANT:"

# user_tag = "[INST]"
# assistant_tag = "[/INST]"

data_path = "../../data/facts/facts_true_false.csv"
dataset = honesty_function_dataset(data_path, tokenizer, user_tag, assistant_tag)

In [ ]:
honesty_rep_reader = mamba_pipeline.get_directions(
    dataset['train']['data'],
    rep_token=rep_token,
    hidden_layers=hidden_layers,
    n_difference=n_difference,
    train_labels=dataset['train']['labels'],
    direction_method=direction_method,
    batch_size=32,
)

In [ ]:
H_tests = mamba_pipeline(
    dataset['test']['data'],
    rep_token=rep_token,
    hidden_layers=hidden_layers,
    rep_reader=honesty_rep_reader,
    batch_size=32)

In [ ]:
# Display some sample labels
dataset['train']['labels'][0:4]

In [ ]:
# Display some sample data
dataset['train']['data'][0:20]

In [ ]:
# Set up plotting
results = {layer: {} for layer in hidden_layers}
rep_readers_means = {}
rep_readers_means['honesty'] = {layer: 0 for layer in hidden_layers}

# For LatentScan
honest_label = 0
true_label = 0
latent_scan_label_descriptions = {
    (honest_label, true_label): "Honest and True",
    (honest_label, 1 - true_label): "Honest and False",
    (1 - honest_label, true_label): "Dishonest and True",
    (1 - honest_label, 1 - true_label): "Dishonest and False"
}

def flat(H):
    return [h[0] if isinstance(h, np.ndarray) else (h.item() if hasattr(h, 'item') else h) for h in H]

# Calculate results
for layer in hidden_layers:
    results[layer]["honesty"] = flat(H_tests[layer])
    rep_readers_means['honesty'][layer] = np.mean(results[layer]["honesty"])

# Plot results
plt.figure(figsize=(10, 6))
plt.scatter(range(len(results[hidden_layers[0]]["honesty"])), results[hidden_layers[0]]["honesty"], alpha=0.5)
plt.axhline(y=rep_readers_means['honesty'][hidden_layers[0]], color='r', linestyle='-')
plt.title("Honesty Representation Direction")
plt.xlabel("Test Example")
plt.ylabel("Honesty Score")
plt.show()